# AreaHustle Voice Benchmark — Pipeline

**Sahara CodeSwitch Africa Challenge · Track 5**

Evaluates three speech-to-text models on **Nigerian Pidgin-English code-switched artisan dispatch commands** (Lagos gig marketplace use case):

| Model | Role |
|---|---|
| **Intron Sahara v2.5** | Domain-specific African-language API (the model AreaHustle uses in production) |
| **OpenAI Whisper Large-v3** | Global multilingual baseline |
| **Meta MMS-1B** | Open-source multilingual baseline |

Two-tier evaluation:
1. **Acoustic** — WER + CER (jiwer) against ground-truth transcripts.
2. **Downstream** — every transcript goes through Gemini (`gemini-3.5-flash-lite`) slot extraction; `category`, `location`, `budget_ngn` are scored against targets.

## How to run
1. **Runtime → Change runtime type → T4 GPU** (required for Whisper + MMS).
2. Run the **Setup** cell.
3. **Upload your audio** — either option works:
   - **Easiest:** run `compress_audio.py` locally (it zips `benchmarks/audio/` after checking filenames), then drag the resulting `audio.zip` into the Colab file browser root (`/content`). The ingestion cell extracts it automatically.
   - Or: create a `raw_audio` folder in the Colab file browser and upload the individual recordings into it.
4. **Order matters:** files are mapped to samples **alphabetically** — the 1st file alphabetically becomes `sample_01`, the 2nd `sample_02`, etc. Name recordings `note01.m4a` … `note20.m4a` (zero-padded!). The ingestion cell prints the full mapping — **verify it before continuing**.
5. Run the remaining cells top to bottom. The final cell prints the two results tables and downloads `results.json`.

In [1]:
# ============================================================ 1. SETUP
!pip install -q jiwer google-genai soundfile

import json, os, re, subprocess, time
from getpass import getpass
from pathlib import Path

import requests
import soundfile as sf
import numpy as np
import jiwer

RAW_DIR = Path("/content/raw_audio")
PROCESSED_DIR = Path("/content/processed_audio")
RAW_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(exist_ok=True)

import torch
print(f"\nSetup OK. GPU available: {torch.cuda.is_available()} "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — enable T4 GPU!'})")
print(f"Upload your 20 recordings into: {RAW_DIR}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.0 MB/s eta 0:00:00

Setup OK. GPU available: True (Tesla T4)
Upload your 20 recordings into: /content/raw_audio


In [2]:
# ============================================================ 2. GROUND TRUTH DATASET
# 20 Nigerian Pidgin artisan dispatch commands with target intent slots.
# Mirrors benchmarks/dataset.json in the repo (source of truth).

DATASET = [
    {"id": 1,  "filename": "sample_01.wav", "transcript": "Abeg I dey Ikeja underbridge now, my brake dey sound, I need mechanic for 5k.", "target": {"category": "Mechanic", "location": "Ikeja underbridge", "budget_ngn": 5000}},
    {"id": 2,  "filename": "sample_02.wav", "transcript": "My kitchen pipe don burst for Yaba tech side, make plumber come fast fast, I get 4000 naira.", "target": {"category": "Plumber", "location": "Yaba", "budget_ngn": 4000}},
    {"id": 3,  "filename": "sample_03.wav", "transcript": "I need rewinder wey go check my pumping machine for Bariga, my budget na 6k.", "target": {"category": "Rewinder", "location": "Bariga", "budget_ngn": 6000}},
    {"id": 4,  "filename": "sample_04.wav", "transcript": "Electrician dey around Surulere? Light full house but my breaker dey trip, I go pay 7500.", "target": {"category": "Electrician", "location": "Surulere", "budget_ngn": 7500}},
    {"id": 5,  "filename": "sample_05.wav", "transcript": "Who dey do carpentry around Ikorodu garage? My wardrobe door don pull out, 3000 dey ground.", "target": {"category": "Carpenter", "location": "Ikorodu garage", "budget_ngn": 3000}},
    {"id": 6,  "filename": "sample_06.wav", "transcript": "My generator alternator don burn for Festac first gate, I need rewinder urgently for 8k.", "target": {"category": "Rewinder", "location": "Festac", "budget_ngn": 8000}},
    {"id": 7,  "filename": "sample_07.wav", "transcript": "Vulcanizer abeg, my tyre puncture for Maryland roundabout, quick assistance for 2000 naira.", "target": {"category": "Vulcanizer", "location": "Maryland roundabout", "budget_ngn": 2000}},
    {"id": 8,  "filename": "sample_08.wav", "transcript": "I dey Lekki phase 1, AC dey blow hot air, mechanic or AC repairer abeg for 10000 naira.", "target": {"category": "AC repairer", "location": "Lekki phase 1", "budget_ngn": 10000}},
    {"id": 9,  "filename": "sample_09.wav", "transcript": "Make painter come paint one room for Ogba, paint dey ground already, labor na 5k.", "target": {"category": "Painter", "location": "Ogba", "budget_ngn": 5000}},
    {"id": 10, "filename": "sample_10.wav", "transcript": "Panel beater needed for Oshodi bus stop, moto scratch side, I fit pay 12k.", "target": {"category": "Panel beater", "location": "Oshodi bus stop", "budget_ngn": 12000}},
    {"id": 11, "filename": "sample_11.wav", "transcript": "Welder dey this area? Iron gate hinge don cut for Ojuelegba, 4500 naira dey.", "target": {"category": "Welder", "location": "Ojuelegba", "budget_ngn": 4500}},
    {"id": 12, "filename": "sample_12.wav", "transcript": "Plumber abeg come fix my WC sink for Ilupeju, water dey leak everywhere, I get 5000.", "target": {"category": "Plumber", "location": "Ilupeju", "budget_ngn": 5000}},
    {"id": 13, "filename": "sample_13.wav", "transcript": "Generator rewinder needed for Victoria Island, starter coil don bad, 15k available.", "target": {"category": "Rewinder", "location": "Victoria Island", "budget_ngn": 15000}},
    {"id": 14, "filename": "sample_14.wav", "transcript": "My ceiling fan motor dey make heavy noise for Mushin, electrician abeg, 3500.", "target": {"category": "Electrician", "location": "Mushin", "budget_ngn": 3500}},
    {"id": 15, "filename": "sample_15.wav", "transcript": "I need bricklayer to patch small soakaway crack for Agege, I go pay 6000 naira.", "target": {"category": "Bricklayer", "location": "Agege", "budget_ngn": 6000}},
    {"id": 16, "filename": "sample_16.wav", "transcript": "Car no wan start for Gbagada expressway, mechanic come with jumper wire, 4k dey.", "target": {"category": "Mechanic", "location": "Gbagada expressway", "budget_ngn": 4000}},
    {"id": 17, "filename": "sample_17.wav", "transcript": "Carpenter needed to fix four office chairs for Marina Lagos island, budget na 8000.", "target": {"category": "Carpenter", "location": "Marina", "budget_ngn": 8000}},
    {"id": 18, "filename": "sample_18.wav", "transcript": "Roofer abeg come check my leaking zinc roof for Ketu, rain dey enter parlour, I get 9000.", "target": {"category": "Roofer", "location": "Ketu", "budget_ngn": 9000}},
    {"id": 19, "filename": "sample_19.wav", "transcript": "Make tailor come sew five school uniform for Egbeda, material don buy already, na 7000 I go pay.", "target": {"category": "Tailor", "location": "Egbeda", "budget_ngn": 7000}},
    {"id": 20, "filename": "sample_20.wav", "transcript": "Fridge repairer needed for Berger, my freezer no dey freeze again, 11000 dey ground.", "target": {"category": "Fridge repairer", "location": "Berger", "budget_ngn": 11000}},
]

print(f"Dataset loaded: {len(DATASET)} samples, language tagged 'pcm'.")

Dataset loaded: 20 samples, language tagged 'pcm'.


In [13]:
# ============================================================ 3. API KEYS
# Enter your keys when prompted (input is hidden). Nothing is stored in the notebook.
SAHARA_API_KEY = getpass("SAHARA_API_KEY: ")

gemini_keys_input = getpass("GEMINI_API_KEYS (Enter up to 3 keys separated by commas): ")
GEMINI_API_KEYS = [k.strip() for k in gemini_keys_input.split(",") if k.strip()]

# Fallback to single key if only one is provided without comma
if not GEMINI_API_KEYS and gemini_keys_input:
    GEMINI_API_KEYS = [gemini_keys_input.strip()]

assert SAHARA_API_KEY and GEMINI_API_KEYS, "Both Sahara and at least one Gemini key are required."
print(f"Keys set. Loaded {len(GEMINI_API_KEYS)} Gemini API key(s) for rotation.")

SAHARA_API_KEY: ··········
GEMINI_API_KEYS (Enter up to 3 keys separated by commas): ··········
Keys set. Loaded 3 Gemini API key(s) for rotation.


In [4]:
# ============================================================ 4. AUDIO INGESTION & STANDARDIZATION
# Accepts EITHER: (a) a zip uploaded straight into /content (drag it into the
# file browser root — no need to create a folder), or (b) loose audio files
# placed inside /content/raw_audio/. Zips are extracted automatically.
# Every file is converted to 16,000 Hz, 16-bit PCM, mono WAV with peak volume
# normalization, saved as sample_01.wav .. sample_20.wav.
# Files are mapped to sample IDs ALPHABETICALLY.

import zipfile

# Auto-extract any zip uploaded to /content into raw_audio/.
for z in Path("/content").glob("*.zip"):
    print(f"Extracting {z.name} ...")
    with zipfile.ZipFile(z) as zf:
        zf.extractall(RAW_DIR)
    print(f"  extracted {len(zf.namelist())} entries")

AUDIO_EXTS = {".m4a", ".aac", ".opus", ".mp3", ".ogg", ".wav", ".amr", ".3gp", ".mp4", ".mpga", ".wave"}

def collect_audio():
    """All audio files under raw_audio, including subfolders (e.g. audio/ from the zip)."""
    return sorted(
        [f for f in RAW_DIR.rglob("*") if f.is_file() and f.suffix.lower() in AUDIO_EXTS],
        key=lambda f: f.name.lower(),
    )

raw_files = collect_audio()
assert raw_files, f"No audio files found in {RAW_DIR}. Upload a zip into /content or files into {RAW_DIR}."
if len(raw_files) != len(DATASET):
    print(f"WARNING: found {len(raw_files)} files but dataset has {len(DATASET)} samples.")
n = min(len(raw_files), len(DATASET))

def standardize(src: Path, dst: Path):
    """ffmpeg transcode to 16kHz mono, then numpy peak-normalize to 16-bit WAV."""
    tmp = dst.with_suffix(".tmp.wav")
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src), "-ar", "16000", "-ac", "1", "-f", "wav", str(tmp)],
        check=True, capture_output=True,
    )
    audio, sr = sf.read(tmp, dtype="float32")
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio * (0.95 / peak)   # peak normalization for consistent loudness
    sf.write(dst, audio, sr, subtype="PCM_16")
    tmp.unlink()

print(f"\n{'Raw file':<40} -> {'Standardized as':<18} {'Duration'}")
print("-" * 75)
durations = {}
for i in range(n):
    sample = DATASET[i]
    dst = PROCESSED_DIR / sample["filename"]
    standardize(raw_files[i], dst)
    durations[sample["filename"]] = round(len(sf.read(dst)[0]) / 16000, 1)
    print(f"{raw_files[i].name:<40} -> {sample['filename']:<18} {durations[sample['filename']]}s")

print("\n>>> VERIFY the mapping above before continuing! <<<")
print(f"Total audio: {sum(durations.values()):.1f}s, average clip {sum(durations.values())/len(durations):.1f}s")

Extracting audio.zip ...
  extracted 20 entries

Raw file                                 -> Standardized as    Duration
---------------------------------------------------------------------------
sample_01.wav                            -> sample_01.wav      5.6s
sample_02.wav                            -> sample_02.wav      5.5s
sample_03.wav                            -> sample_03.wav      5.0s
sample_04.wav                            -> sample_04.wav      7.1s
sample_05.wav                            -> sample_05.wav      7.9s
sample_06.wav                            -> sample_06.wav      8.5s
sample_07.wav                            -> sample_07.wav      10.0s
sample_08.wav                            -> sample_08.wav      9.8s
sample_09.wav                            -> sample_09.wav      7.4s
sample_10.wav                            -> sample_10.wav      8.0s
sample_11.wav                            -> sample_11.wav      7.3s
sample_12.wav                            -> sample_12.

In [5]:
# ============================================================ 5. MODEL 1 — INTRON SAHARA v2.5 (API)
# Sync endpoint, language 'pcm'. Auto-retries when the model is still
# loading ("language not available, wait 30 seconds") up to 3 times.

SAHARA_ENDPOINT = "https://infer.voice.intron.io/file/v1/upload/sync"

def transcribe_sahara(audio_bytes: bytes, api_key: str, lang: str = "pcm",
                      max_retries: int = 3) -> str:
    for attempt in range(max_retries + 1):
        res = requests.post(
            SAHARA_ENDPOINT,
            headers={"Authorization": f"Bearer {api_key}"},
            data={"audio_file_name": "clip.wav", "use_language_asr_input": lang},
            files={"audio_file_blob": ("clip.wav", audio_bytes, "audio/wav")},
            timeout=120,
        )
        body = res.text.lower()
        if "language not available" in body or "wait 30" in body:
            if attempt < max_retries:
                print(f"    model loading, waiting 30s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(30)
                continue
            raise RuntimeError("Sahara model still loading after retries")
        res.raise_for_status()
        return res.json().get("data", {}).get("audio_transcript", "")
    raise RuntimeError("unreachable")

sahara_transcripts = {}
for sample in DATASET:
    path = PROCESSED_DIR / sample["filename"]
    if not path.exists():
        continue
    print(f"[{sample['id']:02d}/20] {sample['filename']} ...", end=" ")
    try:
        sahara_transcripts[sample["filename"]] = transcribe_sahara(path.read_bytes(), SAHARA_API_KEY)
        print(f"OK: {sahara_transcripts[sample['filename']][:60]}")
    except Exception as e:
        sahara_transcripts[sample["filename"]] = ""
        print(f"ERROR: {e}")
    time.sleep(1)  # gentle on the API

print(f"\nSahara done: {sum(1 for v in sahara_transcripts.values() if v)}/20 transcripts")

[01/20] sample_01.wav ... OK: I dey i dey ikeja under bridge my bridge dey sound i need me
[02/20] sample_02.wav ... OK: My kitchen pipe don burst i need plumber fast fast i get 5 k
[03/20] sample_03.wav ... OK: I need rwanda wey go check my pumping machine my budget na 6
[04/20] sample_04.wav ... OK: Electrician dey around sure light dey full house but my brea
[05/20] sample_05.wav ... OK: Who dey do carpenter around di korodo garage? My kitchen war
[06/20] sample_06.wav ... OK: My generator alternator don burn for 1st tack 1st gait and i
[07/20] sample_07.wav ... OK: Vocaniser abeg my tire puncture for maryland rada i need qui
[08/20] sample_08.wav ... OK: I dey lecky face one ac dey blow hot air mechanic call acros
[09/20] sample_09.wav ... OK: Make peter come paint one room for me paint dey ground ready
[10/20] sample_10.wav ... OK: Panemita needed for oshodi bus stop motor crash side afik 12
[11/20] sample_11.wav ... OK: Welder dey dis area iron gate in don cut for Uju lego 4500 n

In [7]:
display(sahara_transcripts)

{'sample_01.wav': 'I dey i dey ikeja under bridge my bridge dey sound i need mechanic for five.\n',
 'sample_02.wav': 'My kitchen pipe don burst i need plumber fast fast i get 5 k for hand\n',
 'sample_03.wav': 'I need rwanda wey go check my pumping machine my budget na 6.\n',
 'sample_04.wav': 'Electrician dey around sure light dey full house but my breaker dey trip i get like 5 k for hand\n',
 'sample_05.wav': 'Who dey do carpenter around di korodo garage? My kitchen wardrobe don fall i get like 5000 naira to give\n',
 'sample_06.wav': 'My generator alternator don burn for 1st tack 1st gait and i need reminder urgently for 8k\n',
 'sample_07.wav': 'Vocaniser abeg my tire puncture for maryland rada i need quick assistance for 2000 naira abeg\n',
 'sample_08.wav': 'I dey lecky face one ac dey blow hot air mechanic call across 10000 naira\n',
 'sample_09.wav': 'Make peter come paint one room for me paint dey ground ready labor na 5k\n',
 'sample_10.wav': 'Panemita needed for oshodi bus 

In [6]:
# ============================================================ 6. MODEL 2 — OPENAI WHISPER LARGE-v3 (local GPU)
# Hugging Face pipeline, default English/multilingual decoding.

from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading openai/whisper-large-v3 on {device} (first load downloads ~3GB)...")
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device=device,
    chunk_length_s=30,
)

whisper_transcripts = {}
for sample in DATASET:
    path = PROCESSED_DIR / sample["filename"]
    if not path.exists():
        continue
    audio, sr = sf.read(path, dtype="float32")
    out = whisper_pipe({"array": audio, "sampling_rate": sr})
    whisper_transcripts[sample["filename"]] = out["text"].strip()
    print(f"[{sample['id']:02d}/20] {whisper_transcripts[sample['filename']][:70]}")

del whisper_pipe
torch.cuda.empty_cache()
print(f"\nWhisper done: {len(whisper_transcripts)} transcripts")

Loading openai/whisper-large-v3 on cuda (first load downloads ~3GB)...


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

...

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface

[01/20] ID Ikeza under bridge. My break day sound. I need mechanic for 5K.
[02/20] My kitchen pipe don't burst. I need plumber fast fast. I get 5K for ha
[03/20] I need rewind. I will go check my pumping machine. My budget nasiski.
[04/20] Electric Shandy around 3 Liri, lights day 4 hours, but my breaker the 
[05/20] Who di do capping ka around the Korodo garage? My kitchen wardrobe don
[06/20] My generator alternator don't burn for first stack, first gate and I n
[07/20] Rokanaisa abeg. My tire puncture for Maryland ride about. I need quick
[08/20] ID Lucky Fish 1 ACD Blue Oats Air Mechanical AC Repairer for 10,000 Na
[09/20] May Peter come paint one room for me. Paint the ground already, live o


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[10/20] Panebita anided fwo osho di bus stop. Motos crash side afipi 2FK.
[11/20] Wè dà dè dì dì sè wè, a yon gè tì ndòn kòt fò o djù e lèk bà wù, 4,500
[12/20] Plumba abe konfis my WC sing for Lugbeju. Wata de lik for eviwe. I get
[13/20] Generator rewinder needed for Vidoria Island. Starter coil done bad. 1
[14/20] My ceiling fan motor, they make heavy noise for motion. Electrician, I
[15/20] I need bricklayer to pass more soccer with craft for a giggy. I go pay
[16/20] Kanu 1 start for Bagada Expressway. Mechanic come with your jumper wir
[17/20] Kapita needed to fix our four office chairs for Marina Lagos Island. B
[18/20] Rufa, I beg, come check my leaking zinc roof for K2. Rain the entire p
[19/20] Mik te lo kom so fai sku uniform sfo ekbeda. Material don dey ready na
[20/20] Food repair are needed for beggar. My freezer know they work again. 11

Whisper done: 20 transcripts


In [8]:
# ============================================================ 7. MODEL 3 — META MMS-1B (local GPU)
# facebook/mms-1b-all with the Nigerian Pidgin ('pcm') adapter,
# falling back to English if unavailable.

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading facebook/mms-1b-all on {device}...")
mms_processor = Wav2Vec2Processor.from_pretrained("facebook/mms-1b-all")
mms_model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/mms-1b-all",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

mms_lang = "pcm"
try:
    mms_processor.tokenizer.set_target_lang("pcm")
    mms_model.load_adapter("pcm")
    print("Using MMS adapter: pcm (Nigerian Pidgin)")
except Exception as e:
    print(f"pcm adapter unavailable ({e}) — falling back to eng")
    mms_processor.tokenizer.set_target_lang("eng")
    mms_model.load_adapter("eng")
    mms_lang = "eng"

mms_transcripts = {}
for sample in DATASET:
    path = PROCESSED_DIR / sample["filename"]
    if not path.exists():
        continue
    audio, sr = sf.read(path, dtype="float32")
    inputs = mms_processor(audio, sampling_rate=sr, return_tensors="pt")
    inputs = {k: (v.half().to(device) if device == "cuda" and v.dtype == torch.float32 else v.to(device))
              for k, v in inputs.items()}
    with torch.no_grad():
        logits = mms_model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    mms_transcripts[sample["filename"]] = mms_processor.batch_decode(pred_ids)[0].strip()
    print(f"[{sample['id']:02d}/20] {mms_transcripts[sample['filename']][:70]}")

del mms_model
torch.cuda.empty_cache()
print(f"\nMMS done: {len(mms_transcripts)} transcripts (adapter: {mms_lang})")

Loading facebook/mms-1b-all on cuda...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.pcm.safetensors: reconstructing file:   0%|          |  0.00B / 8.84MB            

adapter.pcm.safetensors: downloading bytes:           |  0.00B            

Using MMS adapter: pcm (Nigerian Pidgin)
[01/20] idey e kerte ar under brish my breag dey sound and ian mekanik for fiv
[02/20] my kishing py plone boast i nid plonbar fast-fast i gave five key for 
[03/20] i nid rewinder wey go shek my pumping mashine my budget na siskey
[04/20] electri shan dey aloung thru lhree lite dey full house but my beake de
[05/20] who de do kappinkaaroun di korodo garrage my kishin wod rope don fall 
[06/20] my generitor olturnetor don born for first tork face gate and i nid re
[07/20] volkanaizor arbeg my tya ponture for mary landrad about i nid kwiek as
[08/20] i dey lekeifese 1ne acede bluw at a mekanik al ise reparer for 10 000 
[09/20] make pita kon pint one room for mi pint dey ground ared di liborna fiv
[10/20] panebita nieded for hosho di bostob mutus cratsh side afi pitrefke
[11/20] welda dey dis aria iongate eng don kut for jwele bao 4 0005 hdrd nera 
[12/20] plomba abel kon fix my duce sing for lukwe ju wota dey leak for evrywh
[13/20] generator re

In [9]:
# ============================================================ 8. ACOUSTIC METRICS — WER & CER
# Normalization per evaluation spec: lowercase, strip punctuation
# (commas, periods, question marks), collapse whitespace.

def normalize_text(text: str) -> str:
    text = (text or "").lower()
    text = re.sub(r"[,\.,\?]+", " ", text)   # strip punctuation per spec
    return re.sub(r"\s+", " ", text).strip()

MODELS = {
    "sahara_v2_5": sahara_transcripts,
    "whisper_large_v3": whisper_transcripts,
    "mms_1b": mms_transcripts,
}

acoustic = {m: {"wer": [], "cer": []} for m in MODELS}
for sample in DATASET:
    ref = normalize_text(sample["transcript"])
    for model, transcripts in MODELS.items():
        hyp = normalize_text(transcripts.get(sample["filename"], ""))
        if ref and sample["filename"] in transcripts:
            acoustic[model]["wer"].append(jiwer.wer(ref, hyp))
            acoustic[model]["cer"].append(jiwer.cer(ref, hyp))

print("Mean WER / CER per model:\n")
for model, scores in acoustic.items():
    if scores["wer"]:
        print(f"  {model:<18} WER {100 * sum(scores['wer']) / len(scores['wer']):5.1f}%   CER {100 * sum(scores['cer']) / len(scores['cer']):5.1f}%   (n={len(scores['wer'])})")

Mean WER / CER per model:

  sahara_v2_5        WER  33.9%   CER  18.9%   (n=20)
  whisper_large_v3   WER  67.6%   CER  30.2%   (n=20)
  mms_1b             WER  79.9%   CER  36.2%   (n=20)


In [14]:
# ============================================================ 9. DOWNSTREAM SLOT-FILLING — GEMINI 3.5 FLASH-LITE
# Every transcript from every model is parsed into {category, location,
# budget_ngn} by the same Gemini prompt, then scored against targets.
# Implements API key rotation across the provided Gemini keys to bypass 429 errors.

from pydantic import BaseModel, Field
from google import genai

GEMINI_MODEL = "gemini-3.5-flash-lite"

class Slots(BaseModel):
    category: str = Field(description="The trade/artisan type requested, e.g. 'mechanic', 'plumber', 'vulcanizer'")
    location: str = Field(description="The Lagos area/neighbourhood mentioned")
    budget_ngn: float = Field(description="The fee offered in Naira, 0 if not mentioned")

# Initialize a client list from our rotated keys
gemini_clients = [genai.Client(api_key=key) for key in GEMINI_API_KEYS]
current_client_idx = 0

def extract_slots(transcript: str):
    global current_client_idx
    prompt = (
        "Extract the structured intent from this Nigerian Pidgin-English gig request transcript. "
        "The transcript may be noisy or corrupted — extract what is actually there, guess nothing. "
        "Budgets may be phrased as '5k', '5000 naira', '5 bags' — convert to a Naira number. "
        "Use an empty string for category/location and 0 for budget if not present.\n\n"
        f"Transcript: \"{transcript}\""
    )

    num_keys = len(gemini_clients)
    for attempt in range(num_keys):
        client_idx = (current_client_idx + attempt) % num_keys
        client = gemini_clients[client_idx]
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config={"response_mime_type": "application/json", "response_schema": Slots, "temperature": 0.1},
            )
            # Update the starting index for the next call to distribute load
            current_client_idx = client_idx
            return response.parsed
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower() or "limit" in str(e).lower():
                print(f"    [Gemini Client {client_idx + 1}/{num_keys}] Rate limit hit. Rotating to next API key...")
                continue
            else:
                # If it's a non-rate-limit exception, raise it
                raise e

    # If all keys failed due to 429 rate limit error
    raise RuntimeError("All configured Gemini API keys failed due to rate limits or errors.")

def norm_slot(value: str) -> str:
    return normalize_text(str(value or ""))

def slot_match(predicted: str, target: str) -> bool:
    p, t = norm_slot(predicted), norm_slot(target)
    return bool(p) and bool(t) and (p == t or t in p or p in t)

def budget_match(predicted, target: int) -> bool:
    try:
        p = int(round(float(predicted or 0)))
    except (TypeError, ValueError):
        return False
    return p == target or abs(p - target) <= max(100, 0.05 * target)

slot_scores = {m: {"category": [], "location": [], "budget": [], "e2e": []} for m in MODELS}
slot_details = []

for sample in DATASET:
    fname = sample["filename"]
    if fname not in {f for t in MODELS.values() for f in t}:
        continue
    target = sample["target"]
    for model, transcripts in MODELS.items():
        if fname not in transcripts:
            continue
        try:
            slots = extract_slots(transcripts[fname])
            cat_ok = slot_match(slots.category, target["category"])
            loc_ok = slot_match(slots.location, target["location"])
            bud_ok = budget_match(slots.budget_ngn, target["budget_ngn"])
        except Exception as e:
            print(f"  [{model}] {fname}: Gemini error {e}")
            cat_ok = loc_ok = bud_ok = False
            slots = None
        slot_scores[model]["category"].append(cat_ok)
        slot_scores[model]["location"].append(loc_ok)
        slot_scores[model]["budget"].append(bud_ok)
        slot_scores[model]["e2e"].append(cat_ok and loc_ok and bud_ok)
        slot_details.append({
            "filename": fname, "model": model,
            "transcript": transcripts[fname],
            "predicted": slots.model_dump() if slots else None,
            "target": target,
            "scores": {"category": cat_ok, "location": loc_ok, "budget_ngn": bud_ok,
                       "end_to_end": bool(cat_ok and loc_ok and bud_ok)},
        })
    print(f"[{sample['id']:02d}/20] {fname} scored across {len(MODELS)} models")

print("\nDownstream slot accuracy per model:\n")
for model, s in slot_scores.items():
    if s["category"]:
        pct = lambda v: 100 * sum(v) / len(v)
        print(f"  {model:<18} category {pct(s['category']):5.1f}%   location {pct(s['location']):5.1f}%   "
              f"budget {pct(s['budget']):5.1f}%   E2E {pct(s['e2e']):5.1f}%")

[01/20] sample_01.wav scored across 3 models
[02/20] sample_02.wav scored across 3 models
[03/20] sample_03.wav scored across 3 models
[04/20] sample_04.wav scored across 3 models
[05/20] sample_05.wav scored across 3 models
    [Gemini Client 1/3] Rate limit hit. Rotating to next API key...
[06/20] sample_06.wav scored across 3 models
[07/20] sample_07.wav scored across 3 models
[08/20] sample_08.wav scored across 3 models
[09/20] sample_09.wav scored across 3 models
[10/20] sample_10.wav scored across 3 models
    [Gemini Client 2/3] Rate limit hit. Rotating to next API key...
[11/20] sample_11.wav scored across 3 models
[12/20] sample_12.wav scored across 3 models
[13/20] sample_13.wav scored across 3 models
[14/20] sample_14.wav scored across 3 models
[15/20] sample_15.wav scored across 3 models
[16/20] sample_16.wav scored across 3 models
[17/20] sample_17.wav scored across 3 models
[18/20] sample_18.wav scored across 3 models
[19/20] sample_19.wav scored across 3 models
[20/20] s

In [15]:
# ============================================================ 10. FINAL TABLES + results.json
# Prints the two publication tables (markdown — paste straight into the
# 3-page report) and downloads results.json.

MODEL_META = {
    "sahara_v2_5": ("Intron Sahara v2.5", "Domain-Specific API", "Proprietary"),
    "whisper_large_v3": ("OpenAI Whisper Large-v3", "General Transformer", "1.55B"),
    "mms_1b": ("Meta MMS-1B", "Multilingual CTC", "1.0B"),
}

def pct(values):
    return round(100 * sum(values) / len(values), 1) if values else None

aggregate = {}
for model in MODELS:
    a, s = acoustic[model], slot_scores[model]
    if not a["wer"] and not s["category"]:
        continue
    aggregate[model] = {
        "model_name": MODEL_META[model][0],
        "wer_pct": round(100 * sum(a["wer"]) / len(a["wer"]), 1) if a["wer"] else None,
        "cer_pct": round(100 * sum(a["cer"]) / len(a["cer"]), 1) if a["cer"] else None,
        "category_accuracy_pct": pct(s["category"]),
        "location_accuracy_pct": pct(s["location"]),
        "budget_accuracy_pct": pct(s["budget"]),
        "end_to_end_success_pct": pct(s["e2e"]),
        "n_samples": len(s["category"]) or len(a["wer"]),
    }

results = {
    "metadata": {
        "language": "pcm",
        "task": "Nigerian Pidgin-English code-switched artisan dispatch commands",
        "n_samples": len(DATASET),
        "audio_format": "16000 Hz, 16-bit PCM, mono WAV, peak-normalized",
        "gemini_model": GEMINI_MODEL,
        "mms_adapter": mms_lang,
        "durations_s": durations,
    },
    "aggregate": aggregate,
    "per_sample": slot_details,
    "ground_truth": [{"filename": s["filename"], "transcript": s["transcript"], "target": s["target"]} for s in DATASET],
}

with open("/content/results.json", "w", encoding="utf-8") as fh:
    json.dump(results, fh, ensure_ascii=False, indent=2)

print("=" * 80)
print("TABLE 1 — ACOUSTIC PERFORMANCE (SPEECH-TO-TEXT)")
print("=" * 80)
print("| Model | Model Type | Parameters | Dialect | WER (%) | CER (%) |")
print("|---|---|---|---|---|---|")
for model, agg in aggregate.items():
    print(f"| **{agg['model_name']}** | {MODEL_META[model][1]} | {MODEL_META[model][2]} | pcm | "
          f"**{agg['wer_pct']}** | **{agg['cer_pct']}** |")

print()
print("=" * 80)
print("TABLE 2 — DOWNSTREAM AGENTIC TASK PERFORMANCE (GEMINI SLOT-FILLING)")
print("=" * 80)
print("| Transcript Source | Trade Category Acc. (%) | Location Acc. (%) | Budget Acc. (%) | End-to-End Task Success (%) |")
print("|---|---|---|---|---|")
for model, agg in aggregate.items():
    print(f"| **{agg['model_name']}** | {agg['category_accuracy_pct']} | {agg['location_accuracy_pct']} | "
          f"{agg['budget_accuracy_pct']} | **{agg['end_to_end_success_pct']}** |")

print()
print(f"results.json written ({len(slot_details)} per-sample records). Starting download...")
from google.colab import files
files.download("/content/results.json")

TABLE 1 — ACOUSTIC PERFORMANCE (SPEECH-TO-TEXT)
| Model | Model Type | Parameters | Dialect | WER (%) | CER (%) |
|---|---|---|---|---|---|
| **Intron Sahara v2.5** | Domain-Specific API | Proprietary | pcm | **33.9** | **18.9** |
| **OpenAI Whisper Large-v3** | General Transformer | 1.55B | pcm | **67.6** | **30.2** |
| **Meta MMS-1B** | Multilingual CTC | 1.0B | pcm | **79.9** | **36.2** |

TABLE 2 — DOWNSTREAM AGENTIC TASK PERFORMANCE (GEMINI SLOT-FILLING)
| Transcript Source | Trade Category Acc. (%) | Location Acc. (%) | Budget Acc. (%) | End-to-End Task Success (%) |
|---|---|---|---|---|
| **Intron Sahara v2.5** | 55.0 | 40.0 | 80.0 | **15.0** |
| **OpenAI Whisper Large-v3** | 80.0 | 20.0 | 80.0 | **20.0** |
| **Meta MMS-1B** | 60.0 | 30.0 | 50.0 | **5.0** |

results.json written (60 per-sample records). Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>